## Run complete fund timeseries analysis (batch)

For every fund with a Lipper monthly return in the Timeseries sheet, this notebook:

1. Runs the same check as `check-fund-timeseries.ipynb` (sources, **per-fund observation-date resolution**, consistency, regression, Lipper Funds stats comparison, Consolidation sheet replication, worst mismatch). The Excel export includes a **Shifted dates** column on the Lipper vs Yahoo group.
2. Optionally calls OpenAI (`INCLUDE_LLM_CALL`) and saves `outputs/llm-evaluation-openai.json`
3. Optionally writes `outputs/fund-timeseries-report.pptx` per fund (`INCLUDE_POWERPOINT_REPORT`)
4. Optionally merges those slides into `output-data/timeseries-checks/all-funds-report.pptx`
5. Optionally writes a consolidated Excel stub (`INCLUDE_EXCEL_REPORT`) at `output-data/CLEAN_RECOLTE DONNEES-EXCEL_timeseries-checks.xlsx`

**Toggles (first code cell):** `INCLUDE_LLM_CALL` skips API calls and does not require cached LLM JSON when reusing checks. `INCLUDE_POWERPOINT_REPORT` skips per-fund decks and the combined deck. `REUSE_CACHED_OUTPUTS` reuses on-disk checks when they satisfy the flags above.

**Warning:** ~550 funds means many Yahoo/FinanceDatabase downloads and one OpenAI call per fund when LLM is on. Set `FUND_LIMIT` to a small number for a trial run.

In [1]:
INCLUDE_LLM_CALL = False
INCLUDE_POWERPOINT_REPORT = False
INCLUDE_EXCEL_REPORT = True
REUSE_CACHED_OUTPUTS = False
FUND_LIMIT = None


### Setup

Import helpers, reload the batch runner (so code changes apply without restarting the kernel), and define paths to the clean workbook and OpenFIGI cache. Run this once before loading funds or starting the batch.

In [2]:
import sys
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv


def _ensure_helpers_importable() -> None:
    start = Path.cwd().resolve()
    for directory in [start, *start.parents]:
        if (directory / "helpers").is_dir():
            root = str(directory)
            if root not in sys.path:
                sys.path.insert(0, root)
            return
    raise FileNotFoundError(f"Could not find helpers/, starting from {start}")


_ensure_helpers_importable()
load_dotenv(find_dotenv(usecwd=True))

import importlib

import helpers.timeseries_functions.run_fund_batch as _run_fund_batch

importlib.reload(_run_fund_batch)

from helpers.timeseries_functions.load_market_identifiers import load_market_identifiers
from helpers.timeseries_functions.load_workbook import find_workbook, load_workbook

BatchCheckConfig = _run_fund_batch.BatchCheckConfig
combine_fund_reports = _run_fund_batch.combine_fund_reports
default_all_funds_report_path = _run_fund_batch.default_all_funds_report_path
funds_with_timeseries = _run_fund_batch.funds_with_timeseries
run_fund_batch = _run_fund_batch.run_fund_batch

_required_batch_params = {"reuse_cached_outputs", "write_powerpoint"}
if not _required_batch_params.issubset(run_fund_batch.__code__.co_varnames):
    raise RuntimeError(
        "Stale run_fund_batch helper: restart the kernel and run this cell again."
    )

FILENAME = "CLEAN_RECOLTE DONNEES-EXCEL.xlsx"
RELATIVE = Path("input-data-alix") / "clean"
OPENFIGI_CACHE = Path("cache") / "cache-isin-openfigi.json"

### Load funds for the batch

Read the Timeseries sheet, attach OpenFIGI market identifiers, and keep funds with at least one Lipper return. `FUND_LIMIT` truncates the list for trial runs; the result is `fund_isins`, the queue passed to the batch runner.

In [3]:
workbook = find_workbook(RELATIVE, FILENAME)
timeseries_df = load_workbook(RELATIVE, FILENAME)
funds = load_market_identifiers(timeseries_df, OPENFIGI_CACHE)
eligible = funds_with_timeseries(timeseries_df, funds)

if FUND_LIMIT is not None:
    eligible = eligible.head(int(FUND_LIMIT))

fund_isins = eligible["ISIN"].astype(str).tolist()
print(f"Funds with a Lipper timeseries: {len(fund_isins)}")


Read the list of funds - found 550 funds
Mapped to market identifiers
- Funds with market identifier: 549
- Funds without market identifier: 1
Funds with a Lipper timeseries: 550


### Run the batch

`BatchCheckConfig.date_resolution_sessions` (default 6) controls the maximum session shift when aligning Lipper returns to Yahoo. Cached folders written **before** this change must be regenerated once with `REUSE_CACHED_OUTPUTS = False` so resolved dates and index windows are stored in `fund-timeseries.csv`.


In [4]:
config = BatchCheckConfig()
results = run_fund_batch(
    fund_isins,
    timeseries_df=timeseries_df,
    funds=funds,
    config=config,
    run_llm=INCLUDE_LLM_CALL,
    write_powerpoint=INCLUDE_POWERPOINT_REPORT,
    reuse_cached_outputs=REUSE_CACHED_OUTPUTS,
)

ok = results.loc[results["ok"]]
failed = results.loc[~results["ok"]]
reused = ok.loc[ok["reused"]] if "reused" in ok.columns else ok.iloc[0:0]
print(
    f"Succeeded: {len(ok)} · Reused cache: {len(reused)} · Failed: {len(failed)}"
)
if not failed.empty:
    display(failed[["isin", "name", "error"]])
    display(results.head())

[1/550] US74253Q7887 Principal LargeCap S&P 500 Index Fund;Inst


/Users/olivierdedecker/Documents/02 - Enfants/Alix/Mémoire Alix/data/code/helpers/timeseries_functions/load_financedatabase_timeseries.py:72: FutureWarning: The funds database is no longer maintained (since October 2026) and may be outdated; it stays available as it is.
  catalogue = fd.Funds().select()
2026-10-24 18:13:06 - financetoolkit - INFO - Obtaining treasury data for 1 ticker(s)
2026-10-24 18:13:06 - financetoolkit - INFO - Obtaining historical data for 1 ticker(s)
2026-10-24 18:13:06 - financetoolkit - WARNING - No data found for the following tickers: PLFIX
2026-10-24 18:13:07 - financetoolkit - INFO - Obtaining treasury data for 1 ticker(s)
2026-10-24 18:13:07 - financetoolkit - INFO - Obtaining historical data for 1 ticker(s)
/Users/olivierdedecker/Documents/02 - Enfants/Alix/Mémoire Alix/data/code/.venv/lib/python3.13/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


[2/550] US4613088016 American Funds Investment Co of America;529-E


/Users/olivierdedecker/Documents/02 - Enfants/Alix/Mémoire Alix/data/code/helpers/timeseries_functions/load_financedatabase_timeseries.py:72: FutureWarning: The funds database is no longer maintained (since October 2026) and may be outdated; it stays available as it is.
  catalogue = fd.Funds().select()
2026-10-24 18:13:10 - financetoolkit - INFO - Obtaining treasury data for 1 ticker(s)
2026-10-24 18:13:11 - financetoolkit - INFO - Obtaining historical data for 1 ticker(s)
2026-10-24 18:13:11 - financetoolkit - WARNING - No data found for the following tickers: CICEX
2026-10-24 18:13:11 - financetoolkit - INFO - Obtaining treasury data for 1 ticker(s)
2026-10-24 18:13:11 - financetoolkit - INFO - Obtaining historical data for 1 ticker(s)
/Users/olivierdedecker/Documents/02 - Enfants/Alix/Mémoire Alix/data/code/.venv/lib/python3.13/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


[3/550] US3608023004 American Funds Fundamental Investors;C


/Users/olivierdedecker/Documents/02 - Enfants/Alix/Mémoire Alix/data/code/helpers/timeseries_functions/load_financedatabase_timeseries.py:72: FutureWarning: The funds database is no longer maintained (since October 2026) and may be outdated; it stays available as it is.
  catalogue = fd.Funds().select()
2026-10-24 18:13:15 - financetoolkit - INFO - Obtaining treasury data for 1 ticker(s)
2026-10-24 18:13:15 - financetoolkit - INFO - Obtaining historical data for 1 ticker(s)
2026-10-24 18:13:15 - financetoolkit - WARNING - No data found for the following tickers: AFICX
2026-10-24 18:13:15 - financetoolkit - INFO - Obtaining treasury data for 1 ticker(s)
2026-10-24 18:13:16 - financetoolkit - INFO - Obtaining historical data for 1 ticker(s)
/Users/olivierdedecker/Documents/02 - Enfants/Alix/Mémoire Alix/data/code/.venv/lib/python3.13/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


... (1,034 outputs left out)

[550/550] US45259A1593 VistaShares Shield S&P 500 Enhanced Protection ETF
Succeeded: 0 · Reused cache: 0 · Failed: 550


/Users/olivierdedecker/Documents/02 - Enfants/Alix/Mémoire Alix/data/code/helpers/timeseries_functions/load_financedatabase_timeseries.py:72: FutureWarning: The funds database is no longer maintained (since October 2026) and may be outdated; it stays available as it is.
  catalogue = fd.Funds().select()


,isin,name,error
0,US74253Q7887,Principal LargeCap S&P 500 Index Fund;Inst,name 'json_ready' is not defined
1,US4613088016,American Funds Investment Co of America;529-E,name 'json_ready' is not defined
2,US3608023004,American Funds Fundamental Investors;C,name 'json_ready' is not defined
3,US87245M1457,Nuveen S&P 500 Index Fund;I,name 'json_ready' is not defined
4,US78413L6965,SEI Inst Mgd Dynamic Asset Allocation Fund;F,name 'json_ready' is not defined
...,...,...,...
545,US2103227649,TrueShares Structured Outcome ETF - December -...,'DECZ is not in the FinanceDatabase funds cata...
546,US2103226815,TrueShares Structured Outcome ETF - USD,'JANZ is not in the FinanceDatabase funds cata...
547,US2103227722,TrueShares Structured Outcome November ETF - USD,'NOVZ is not in the FinanceDatabase funds cata...
548,US2103227805,TrueShares Structured Outcome October ETF - USD,'OCTZ is not in the FinanceDatabase funds cata...


,isin,name,ok,reused,error,check_dir,report_path
0,US74253Q7887,Principal LargeCap S&P 500 Index Fund;Inst,False,False,name 'json_ready' is not defined,None,None
1,US4613088016,American Funds Investment Co of America;529-E,False,False,name 'json_ready' is not defined,None,None
2,US3608023004,American Funds Fundamental Investors;C,False,False,name 'json_ready' is not defined,None,None
3,US87245M1457,Nuveen S&P 500 Index Fund;I,False,False,name 'json_ready' is not defined,None,None
4,US78413L6965,SEI Inst Mgd Dynamic Asset Allocation Fund;F,False,False,name 'json_ready' is not defined,None,None


### Combine PowerPoint reports

When `INCLUDE_POWERPOINT_REPORT` is True, stitch each successful fund’s `outputs/fund-timeseries-report.pptx` into one deck at `output-data/timeseries-checks/all-funds-report.pptx` (one slide per fund). Otherwise this step is skipped.

In [5]:
if INCLUDE_POWERPOINT_REPORT:
    report_paths = [Path(path) for path in ok["report_path"].dropna()]
    combined_path = default_all_funds_report_path()

    if not report_paths:
        raise RuntimeError(
            "No reports were generated; fix failures above before combining."
        )

    combine_fund_reports(report_paths, combined_path)
    from pptx import Presentation

    deck = Presentation(str(combined_path))
    print(f"Combined deck: {combined_path}")
    print(f"Slides: {len(deck.slides)} (one per successful fund)")
else:
    print("Skipped combined PowerPoint (INCLUDE_POWERPOINT_REPORT is False).")

Skipped combined PowerPoint (INCLUDE_POWERPOINT_REPORT is False).


### Export consolidated Excel

When `INCLUDE_EXCEL_REPORT` is True, write `output-data/CLEAN_RECOLTE DONNEES-EXCEL_timeseries-checks.xlsx` from `output-data/timeseries-checks/<ISIN>/` caches (Timeseries-checks, Regression-exclusions, Methodology). The export uses the same `eligible` table as the batch (respects `FUND_LIMIT`), not the full Lipper list. Close the file in Excel before re-running.
The **Timeseries-checks** sheet includes **Shifted dates** (count of resolved months) and a **Methodology** entry describing per-fund date resolution.


In [6]:
if INCLUDE_EXCEL_REPORT:
    import importlib

    import helpers.timeseries_functions.write_timeseries_checks_excel as _ts_excel

    importlib.reload(_ts_excel)

    excel_path = _ts_excel.export_timeseries_checks(eligible, workbook)
    print(f"Saved {excel_path}")
else:
    print("Skipped Excel export (INCLUDE_EXCEL_REPORT is False).")

/Users/olivierdedecker/Documents/02 - Enfants/Alix/Mémoire Alix/data/code/.venv/lib/python3.13/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


Timeseries-checks export: 456 fund rows written to /Users/olivierdedecker/Documents/02 - Enfants/Alix/Mémoire Alix/data/code/output-data/CLEAN_RECOLTE DONNEES-EXCEL_timeseries-checks.xlsx (Timeseries-checks + Regression-exclusions + Methodology)
Saved /Users/olivierdedecker/Documents/02 - Enfants/Alix/Mémoire Alix/data/code/output-data/CLEAN_RECOLTE DONNEES-EXCEL_timeseries-checks.xlsx
